In [230]:
from sklearn.base import clone
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Model eval

## Load data

### functions

In [ ]:
def get_models(random_state=42):

    return {
        "SVM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=True,
                random_state=random_state
            ))
        ]),
        
        "KNN": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5))
        ]),

        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                random_state=random_state
            ))
        ]),

        "XGBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                eval_metric="logloss",
                random_state=random_state
            ))
        ]),
        "MLP": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=1e-3,
                learning_rate_init=1e-3,
                max_iter=1000,
                early_stopping=True,
                random_state=random_state
            ))
        ]
    )
    }

def compute_metrics_from_predictions(predictions_df, on_columns="language", filter_values=None):
    metrics_rows = []
    per_class_rows = []
    # for col, values in zip(on_columns, filter_values):
    #     if col not in predictions_df.columns:
    #         raise ValueError(f"Column '{col}' not found in predictions DataFrame.")
    #     predictions_df = predictions_df[predictions_df[col].isin(values)]
    if filter_values is not None:
        predictions_df = predictions_df[predictions_df[on_columns].isin(filter_values)]
    group_cols = ["model", "feature_set"]

    for (model_name, feature_set), df_group in predictions_df.groupby(group_cols):
        y_true = df_group["y_true"].values
        y_pred = df_group["y_pred"].values
        prob_class_0 = df_group["prob_class_0"].values
        prob_class_1 = df_group["prob_class_1"].values
        classes = np.sort(np.unique(y_true))

        row = {
            "model": model_name,
            "feature_set": feature_set,
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "auc": roc_auc_score(y_true, prob_class_1) if len(classes) == 2 else None
        }

        metrics_rows.append(row)

        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "class_or_avg"})
        report_df["model"] = model_name
        report_df["feature_set"] = feature_set
        per_class_rows.append(report_df)

    metrics_df = pd.DataFrame(metrics_rows)
    per_class_metrics_df = pd.concat(per_class_rows, ignore_index=True)

    return metrics_df, per_class_metrics_df

In [200]:
# splits
def load_split_df(csv_path):
    split_df = pd.read_csv(csv_path).copy()
    split_df["split"] = split_df["split"].astype(str).str.strip().str.lower()
    return split_df

def make_split_based_subset(
    df,
    csv_path,
    languages,
    split_column="split",
    train_value="train",
    test_value="test",
):
    split_df = load_split_df(csv_path)
    merged_df = pd.merge(df, split_df[['file_id','label', 'split']], on=("file_id",'label'), how="left")
    key_columns = ["file_id", "label"]

    missing_in_features = [c for c in key_columns if c not in df.columns]
    missing_in_split = [c for c in key_columns if c not in split_df.columns]

    if missing_in_features:
        raise ValueError(f"features_df missing key columns: {missing_in_features}")
    if missing_in_split:
        raise ValueError(f"split_df missing key columns: {missing_in_split}")
    if "split" not in split_df.columns:
        raise ValueError("split_df must contain a 'split' column")
    

    subset_df = merged_df[merged_df["language"].isin(languages)].copy()

    if subset_df.empty:
        raise ValueError(
            f"No rows found for languages={languages}. "
            f"Available languages: {sorted(merged_df['language'].dropna().astype(str).unique().tolist())}"
        )

    split_series = subset_df[split_column].astype(str).str.strip().str.lower()

    train_df = subset_df[split_series == str(train_value).lower()].copy()
    test_df = subset_df[split_series == str(test_value).lower()].copy()

    if train_df.empty or test_df.empty:
        raise ValueError(
            f"Split-based subset produced empty train/test. "
            f"Train rows={len(train_df)}, test rows={len(test_df)}, languages={languages}"
        )

    return train_df, test_df

def make_cross_language_split(features_df, train_languages, test_languages):
    train_df = features_df[features_df["language"].isin(train_languages)].copy()
    test_df = features_df[features_df["language"].isin(test_languages)].copy()

    if train_df.empty:
        raise ValueError(f"No train rows found for train_languages={train_languages}")
    if test_df.empty:
        raise ValueError(f"No test rows found for test_languages={test_languages}")

    return train_df, test_df

In [201]:
# table manager
def load_feature_tables(feature_paths_by_name):
    """
    feature_paths_by_name: dict like
    {
        "en": "data/processed/features_en.csv",
        "es": "data/processed/features_es.csv",
        "zh": "data/processed/features_zh.csv",
    }
    """
    tables = {}

    for name, path in feature_paths_by_name.items():
        df = pd.read_csv(path)
        tables[name] = df

    return tables

def subset_by_languages(df, languages):
    return df[df["language"].isin(languages)].copy()

def build_master_feature_table(feature_tables):
    """
    feature_tables: dict of name -> df
    Returns a concatenated master dataframe.
    """
    df = pd.concat(list(feature_tables.values()), ignore_index=True)
    return df

In [202]:
def train_and_predict(
    train_df,
    test_df,
    feature_columns,
    models,
    experiment_name,
    strategy,
    feature_set_name,
    train_languages,
    test_languages,
    target_column="label",
    sample_id_column="file_id",
    language_column="language",
    dataset_column="dataset",
):
    X_train = train_df[feature_columns].copy()
    y_train = train_df[target_column].values

    X_test = test_df[feature_columns].copy()
    y_test = test_df[target_column].values
    prediction_rows = []

    for model_name, model in models.items():
        clf = clone(model)
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)

        row_dict = {
            "sample_id": test_df[sample_id_column].values,
            "language": test_df[language_column].values,
            "dataset": test_df[dataset_column].values if dataset_column in test_df.columns else ["unknown"] * len(test_df),
            "model": model_name,
            "feature_set": [feature_set_name] * len(test_df),
            "experiment_name": [experiment_name] * len(test_df),
            "strategy": [strategy] * len(test_df),
            "train_languages": [",".join(train_languages)] * len(test_df),
            "test_languages": [",".join(test_languages)] * len(test_df),
            "y_true": y_test,
            "y_pred": y_pred,
        }

        if hasattr(clf, "predict_proba"):
            y_proba = clf.predict_proba(X_test)
            for class_idx in range(y_proba.shape[1]):
                row_dict[f"prob_class_{class_idx}"] = y_proba[:, class_idx]

        prediction_rows.append(pd.DataFrame(row_dict))

    return pd.concat(prediction_rows, ignore_index=True)

In [219]:
def run_experiment(
    features_df,
    feature_columns,
    models,
    experiment,
    feature_set_name,
    target_column="label",
):
    strategy = experiment["strategy"]
    print(f"Running experiment: {experiment['name']} with strategy: {strategy}")
    if strategy == "mono":
        language = experiment["language"]
        print(f"Running mono-language experiment for language: {language}")

        train_df, test_df = make_split_based_subset(
            df=features_df,
            csv_path=experiment["csv_path"],
            languages=language
        )

        print(f"Train set size: {len(train_df)}, Test set size: {len(test_df)}")

        return train_and_predict(
            train_df=train_df,
            test_df=test_df,
            feature_columns=feature_columns,
            models=models,
            experiment_name=experiment["name"],
            strategy=strategy,
            feature_set_name=feature_set_name,
            train_languages=language,
            test_languages=language,
            target_column=target_column,
        )

    elif strategy == "cross":

        train_df, test_df = make_cross_language_split(
        features_df,
        train_languages=experiment["train_languages"],
        test_languages=experiment["test_languages"],
        )

        return train_and_predict(
            train_df=train_df,
            test_df=test_df,
            feature_columns=feature_columns,
            models=models,
            experiment_name=experiment["name"],
            strategy=strategy,
            feature_set_name=feature_set_name,
            train_languages=experiment["train_languages"],
            test_languages=experiment["test_languages"],
            target_column=target_column,
        )

    elif strategy == "multi":

        train_df, test_df = make_split_based_subset(
            df=features_df,
            languages=language,
            csv_path=experiment["csv_path"],
        )

        return train_and_predict(
            train_df=train_df,
            test_df=test_df,
            feature_columns=feature_columns,
            models=models,
            experiment_name=experiment["name"],
            strategy=strategy,
            feature_set_name=feature_set_name,
            train_languages=language,
            test_languages=language,
            target_column=target_column,
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

In [204]:
def run_all_experiments(
    features_df,
    feature_sets,
    experiments,
    get_models_fn,
    random_state=42,
):
    all_predictions = []

    for feature_set_name, feature_columns in feature_sets.items():
        print(f"\nRunning feature set: {feature_set_name}")
        models = get_models_fn(random_state=random_state)

        for experiment in experiments:
            print(f"  Experiment: {experiment['name']} [{experiment['strategy']}]")

            preds = run_experiment(
                features_df=features_df,
                feature_columns=feature_columns,
                models=models,
                experiment=experiment,
                feature_set_name=feature_set_name,
                target_column="label",
            )

            all_predictions.append(preds)

    return pd.concat(all_predictions, ignore_index=True)

### running

In [ ]:
EXPERIMENTS = [
        {
        "name": "mono_english",
        "strategy": "mono",
        "language": ["English"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",
    },
    {
        "name": "mono_mandarin",
        "strategy": "mono",
        "language": ["Mandarin"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",
    },
    {
        "name": "train_english_test_mandarin",
        "strategy": "cross",
        "train_languages": ["English"],
        "test_languages": ["Mandarin"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",
    },
    {
        "name": "train_mandarin_english_test_english_mandarin",
        "strategy": "multi",
        "language": ["English", "Mandarin"],
        "csv_path": r"D:\masteruwefduyqeahfdqe\ASR-project\FIN_split_70_30_by_language_grouped_by_safe_speaker.csv",    
    }

]

In [ ]:
FEATURE_SETS = {
    "all": [
        "total_speaking_time_normalized",
        "interviewer_interruptions",
        "word_rate_words_per_sec",
        # "word_rate_overall",
        "mean_utterance_word_rate_words_per_sec",
        "median_utterance_word_rate_words_per_sec",
        "std_utterance_word_rate_words_per_sec",
        "unique_word_count",
        "type_token_ratio",
        "mean_words_per_utterance",
        "median_words_per_utterance",
        "std_words_per_utterance",
        "mean_utterance_duration_sec",
        "median_utterance_duration_sec",
        "std_utterance_duration_sec",
        "MeanUnvoicedSegmentLength",
        "StddevUnvoicedSegmentLength",
        "F0semitoneFrom27.5Hz_sma3nz_amean",
        "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
        "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
        "loudness_sma3_amean",
        "loudness_sma3_stddevNorm",
        "loudness_sma3_pctlrange0-2",
        "loudnessPeaksPerSec",
        "jitterLocal_sma3nz_amean",
        "shimmerLocaldB_sma3nz_amean",
        "HNRdBACF_sma3nz_amean",
        "alphaRatioV_sma3nz_amean",
        "slopeV0-500_sma3nz_amean",
        "slopeV500-1500_sma3nz_amean",
        "equivalentSoundLevel_dBp",
    ],
    "cha_only": [
        "total_speaking_time_normalized",
        "interviewer_interruptions",
        "word_rate_words_per_sec",
    ],
    "wav_only": [
        "MeanUnvoicedSegmentLength",
        "StddevUnvoicedSegmentLength",
        "F0semitoneFrom27.5Hz_sma3nz_amean",
        "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
        "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
        "loudness_sma3_amean",
        "loudness_sma3_stddevNorm",
        "loudness_sma3_pctlrange0-2",
        "loudnessPeaksPerSec",
        "jitterLocal_sma3nz_amean",
        "shimmerLocaldB_sma3nz_amean",
        "HNRdBACF_sma3nz_amean",
        "alphaRatioV_sma3nz_amean",
        "slopeV0-500_sma3nz_amean",
        "slopeV500-1500_sma3nz_amean",
        "equivalentSoundLevel_dBp",
    ],
}

In [207]:
# 1. load feature tables
feature_paths = {
    "English": "full_feature_table_pitts.csv",
    "Mandarin": "full_feature_table_chinese.csv",
    "Greek": "full_feature_table_greek.csv",
}

feature_tables = load_feature_tables(feature_paths)
features_df = build_master_feature_table(feature_tables)

In [220]:
# 2. run experiments
predictions_df = run_all_experiments(
    features_df=features_df,
    feature_sets=FEATURE_SETS,
    experiments=EXPERIMENTS,
    get_models_fn=get_models,
    random_state=42,
)


Running feature set: all
  Experiment: mono_en [mono]
Running experiment: mono_en with strategy: mono
Running mono-language experiment for language: ['English']
Train set size: 390, Test set size: 159
  Experiment: mono_ch [mono]
Running experiment: mono_ch with strategy: mono
Running mono-language experiment for language: ['Mandarin']
Train set size: 178, Test set size: 81
  Experiment: train_en_test_ch [cross]
Running experiment: train_en_test_ch with strategy: cross


In [221]:
# check for NAn values in features_df
nan_counts = features_df.isna().sum()
print("NaN counts in features_df:")
print(nan_counts)

# which rows have NaN values in features_df
nan_rows = features_df[features_df.isna().any(axis=1)]
print("Rows with NaN values in features_df:")
nan_rows[["file_id", "language", "label", "std_utterance_word_rate_words_per_sec","std_words_per_utterance"]]

NaN counts in features_df:
file_id                                     0
wav_path                                    0
cha_path                                    0
label                                       0
language                                    0
total_time_ms                               0
total_speaking_time_ms                      0
total_speaking_time_normalized              0
interviewer_interruptions                   0
word_count                                  0
word_rate_words_per_sec                     0
mean_utterance_word_rate_words_per_sec      0
median_utterance_word_rate_words_per_sec    0
std_utterance_word_rate_words_per_sec       5
unique_word_count                           0
type_token_ratio                            0
n_par_utterances                            0
mean_words_per_utterance                    0
median_words_per_utterance                  0
std_words_per_utterance                     5
mean_utterance_duration_sec                 0
median_

,file_id,language,label,std_utterance_word_rate_words_per_sec,std_words_per_utterance
559,016_Daddy,Mandarin,1,NaN,NaN
660,135_market,Mandarin,1,NaN,NaN
752,025_Daddy,Mandarin,0,NaN,NaN
814,37,Greek,1,NaN,NaN
831,78,Greek,1,NaN,NaN


In [194]:
predictions_df[predictions_df["strategy"] == "cross"]

,sample_id,language,dataset,model,feature_set,experiment_name,strategy,train_languages,test_languages,y_true,y_pred,prob_class_0,prob_class_1
960,006_Daddy,Mandarin,Mandarin_Chou,SVM,all,train_en_test_ch,cross,English,Mandarin,1,1,0.359273,0.640727
961,006_market,Mandarin,Mandarin_Chou,SVM,all,train_en_test_ch,cross,English,Mandarin,1,1,0.329266,0.670734
962,006_park,Mandarin,Mandarin_Chou,SVM,all,train_en_test_ch,cross,English,Mandarin,1,1,0.352408,0.647592
963,011_Daddy,Mandarin,Mandarin_Chou,SVM,all,train_en_test_ch,cross,English,Mandarin,1,1,0.237643,0.762357
964,011_market,Mandarin,Mandarin_Chou,SVM,all,train_en_test_ch,cross,English,Mandarin,1,1,0.308935,0.691065
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1991,064_market,Mandarin,Mandarin_Chou,XGBoost,all,train_en_test_ch,cross,English,Mandarin,0,1,0.043717,0.956283
1992,064_park,Mandarin,Mandarin_Chou,XGBoost,all,train_en_test_ch,cross,English,Mandarin,0,1,0.422534,0.577466
1993,136_Daddy,Mandarin,Mandarin_Chou,XGBoost,all,train_en_test_ch,cross,English,Mandarin,0,0,0.800297,0.199703
1994,136_market,Mandarin,Mandarin_Chou,XGBoost,all,train_en_test_ch,cross,English,Mandarin,0,0,0.523314,0.476686


In [136]:
predictions_df[predictions_df["experiment_name"] == "mono_en"]

,sample_id,language,dataset,model,feature_set,experiment_name,strategy,train_languages,test_languages,y_true,y_pred,prob_class_0,prob_class_1
0,001-0,English,Pitt,SVM,all,mono_en,mono_language,English,English,1,1,0.297327,0.702673
1,014-2,English,Pitt,SVM,all,mono_en,mono_language,English,English,1,1,0.400776,0.599224
2,024-1,English,Pitt,SVM,all,mono_en,mono_language,English,English,1,1,0.330052,0.669948
3,024-2,English,Pitt,SVM,all,mono_en,mono_language,English,English,1,1,0.336035,0.663965
4,043-0,English,Pitt,SVM,all,mono_en,mono_language,English,English,1,0,0.612652,0.387348
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2563,323-0,English,Pitt,XGBoost,wav_only,mono_en,mono_language,English,English,0,1,0.387553,0.612447
2564,323-1,English,Pitt,XGBoost,wav_only,mono_en,mono_language,English,English,0,1,0.352462,0.647538
2565,612-0,English,Pitt,XGBoost,wav_only,mono_en,mono_language,English,English,0,1,0.256106,0.743894
2566,686-0,English,Pitt,XGBoost,wav_only,mono_en,mono_language,English,English,0,1,0.361949,0.638051


In [227]:
metrics_df, per_class_metrics_df = compute_metrics_from_predictions(predictions_df, on_columns="language", filter_values=["English"])

In [228]:
print(predictions_df.head())
print(metrics_df)
print(per_class_metrics_df.head())

  sample_id language dataset model feature_set experiment_name strategy  \
0     014-2  English    Pitt   SVM         all         mono_en     mono   
1     024-1  English    Pitt   SVM         all         mono_en     mono   
2     024-2  English    Pitt   SVM         all         mono_en     mono   
3     043-0  English    Pitt   SVM         all         mono_en     mono   
4     046-0  English    Pitt   SVM         all         mono_en     mono   

  train_languages test_languages  y_true  y_pred  prob_class_0  prob_class_1  
0         English        English       1       1      0.400776      0.599224  
1         English        English       1       1      0.330052      0.669948  
2         English        English       1       1      0.336035      0.663965  
3         English        English       1       0      0.612652      0.387348  
4         English        English       1       0      0.494567      0.505433  
          model feature_set  accuracy  balanced_accuracy  precision_macro  

In [229]:
metrics_df

,model,feature_set,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,auc
0,KNN,all,0.566038,0.565262,0.565014,0.565262,0.564936,0.622973
1,MLP,all,0.641509,0.630604,0.646226,0.630604,0.626376,0.696025
2,RandomForest,all,0.647799,0.639984,0.648232,0.639984,0.638871,0.705962
3,SVM,all,0.660377,0.651749,0.662940,0.651749,0.650293,0.710334
4,XGBoost,all,0.628931,0.622337,0.627232,0.622337,0.621688,0.688871


In [ ]:
exp_features_df["sample_id"] = exp_features_df["wav_path"].apply(lambda x: Path(x).stem)

FEATURE_COLUMNS = [
    "total_speaking_time_normalized",
    "interviewer_interruptions",
    "word_rate_words_per_sec",
]

models = get_models(random_state=42)

predictions_df = generate_cv_predictions(
    features_df=exp_features_df,
    feature_columns=FEATURE_COLUMNS,
    models=models,
    target_column="label",
    sample_id_column="sample_id",
    feature_set_name="basic_interpretable",
    n_splits=5,
    random_state=42
)

metrics_df, per_class_metrics_df = compute_metrics_from_predictions(predictions_df)

print(predictions_df.head())
print(metrics_df)
print(per_class_metrics_df.head())

  sample_id model          feature_set  fold  y_true  y_pred  prob_class_0  \
0     016-0   SVM  basic_interpretable     1       1       0      0.633635   
1     157-2   SVM  basic_interpretable     1       1       0      0.717935   
2     164-2   SVM  basic_interpretable     1       1       0      0.661806   
3     216-0   SVM  basic_interpretable     1       1       0      0.638678   
4     235-2   SVM  basic_interpretable     1       1       0      0.500000   

   prob_class_1  
0      0.366365  
1      0.282065  
2      0.338194  
3      0.361322  
4      0.500000  
          model          feature_set  accuracy  balanced_accuracy  \
0           KNN  basic_interpretable  0.634146           0.633201   
1  RandomForest  basic_interpretable  0.663957           0.663360   
2           SVM  basic_interpretable  0.691057           0.689286   
3       XGBoost  basic_interpretable  0.642276           0.641931   

   precision_macro  recall_macro  f1_macro   roc_auc  
0         0.634068    